# Janus 04 Valuation｜v22 歷史回溯與有效月末定向補收

**此版會真實呼叫 API（`RUN_FETCH=True`），預設每輪先抓 5 組有界請求**。第21版已抓的35份 raw/receipts **只讀不改**，新證據只寫入獨立的 `FINMIND_MIN30_HIST_V22_20261024` 研究目錄；先檢查前五筆新增正值月末，沒有增益則暫停擴大。

**對應前一輪 NO-GO 修正：**不再把 2023-10～2026-09 的 `NULL_MULTIPLE` 與經濟性零／負 PE 當作可補正值；先利用現有36個月，對 PE/PB 未滿30的普通股依當前公司主檔上市日期定向追查 **2021-10～2023-09 的缺少歷史**。上市日期未知、IPO前、原36月幾乎都沒有正 PE／PB 的股票先標 HOLD；不硬補到30。真正 PIT 歷史身分仍未驗證。

**必要輸入：**第20版 `preflight_30_valid_monthends_20261024/` 六份結果仍保留；第21版根路徑已填入，只讀已抓資料。不使用付費API、不變更 canonical、Risk ACTIVE、Registry，`publication_allowed=false`。

**每輪完成後看：**`00_run_summary_v22.json`、`v22_valid_month_gain.csv`、`v22_task_status.csv`、`v22_pairs.csv`。原始FinMind回應及SHA256 receipt會隨每個任務寫入新的 `raw/` 和 `receipts/`。首次起跑先5筆，看到有效增益再調高 `MAX_CALLS_PER_RUN` 後重新「全部執行」。

In [ ]:
# 1. 掛載 Drive；僅授權 janusChatGPT 的研究目錄
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json,hashlib,requests,getpass,os,shutil,time,datetime as dt,sys,types
ROOT=Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
SOURCE=ROOT/'模型研究'/'valuation'/'preflight_30_valid_monthends_20261024'
LEGACY=ROOT/'模型研究'/'valuation'/'backfill_candidates'/'FINMIND_MIN30_REAL_20261024'
DEST=ROOT/'模型研究'/'valuation'/'backfill_candidates'/'FINMIND_MIN30_HIST_V22_20261024'
RUN_FETCH=True             # 真實API請求（將進行最多5筆）；False只盤點
MAX_CALLS_PER_RUN=5        # 第一次僅5筆觀察補收效益；不要直接設700
SLEEP_SECONDS=8.0         # 免費API保守限流；若quota暫停就停止
MAX_LOOKBACK_MONTHS=60    # 2026-09往回60個月，早於2023-10有查詢任務
MIN_EXISTING_POSITIVE=12  # <12月者先HOLD，不浪費請求；與正式30月門檻不同
RETRY_ONCE=False
assert ROOT.is_dir() and ROOT.name=='janusChatGPT' and SOURCE.is_dir()
assert 1<=MAX_CALLS_PER_RUN<=25 and SLEEP_SECONDS>=7.2
assert DEST.resolve().is_relative_to(ROOT.resolve())
print('V22獨立輸出：',DEST)
print('先行HTTP請求上限：',MAX_CALLS_PER_RUN,'真實抓取：',RUN_FETCH)

In [ ]:
# 2. 內嵌經本地測試的 v21 基礎引擎（作為 Module，避免依賴線上GitHub浮動main）
_base_code = '"""Janus Valuation 04 research-only FinMind scoped backfill. No canonical writes."""\nfrom __future__ import annotations\nimport csv\nimport datetime as dt\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport re\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nBASE_URL = \'https://api.finmindtrade.com/api/v4/data\'\nOFFICIAL = {\'twse\': \'https://openapi.twse.com.tw/v1/opendata/t187ap03_L\',\n            \'tpex\': \'https://mopsfin.twse.com.tw/opendata/t187ap03_O.csv\'}\nDATASETS = {\'valuation\': \'TaiwanStockPER\',\n            \'income\': \'TaiwanStockFinancialStatements\',\n            \'balance\': \'TaiwanStockBalanceSheet\'}\nREQUIRED = (\'00_summary.json\', \'01_普通股估值與季報逐股核對.csv\',\n            \'02_PE_PB_30有效月末缺口逐月查核.csv\',\n            \'03_30有效月末_股票分支短缺清單.csv\',\n            \'04_財報無紀錄_相關性待核查.csv\', \'master_sources.json\')\nEXPECTED_RELEASE = \'release-0bbae8d9c1ca7d8d\'\nEXPECTED_CURRENT_SHA = \'0b64ac6927805fb98260121da94624d7e08ac423f0aa13b8349d4c304f0135cd\'\nEXPECTED_MANIFEST_SHA = \'7b80d441efabbc307d359ea5d15cd9202ccee5e2930cf4c3cc337496c95eb542\'\n\n\ndef sha(data: bytes) -> str: return hashlib.sha256(data).hexdigest()\ndef utc(): return dt.datetime.now(dt.timezone.utc).isoformat()\n\ndef atomic_bytes(path: Path, data: bytes):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp=path.with_name(path.name+\'.part\')\n    with open(tmp, \'wb\') as f:\n        f.write(data); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp, path)\n\ndef atomic_json(path: Path, data):\n    atomic_bytes(path, (json.dumps(data, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False)+\'\\n\').encode())\n\ndef records(path: Path):\n    with open(path, \'r\', encoding=\'utf-8-sig\', newline=\'\') as f:\n        return list(csv.DictReader(f))\n\ndef atomic_csv(path: Path, rows: list[dict], fields: list[str]):\n    f=io.StringIO(newline=\'\'); writer=csv.DictWriter(f, fieldnames=fields, extrasaction=\'ignore\')\n    writer.writeheader(); writer.writerows(rows)\n    atomic_bytes(path, f.getvalue().encode(\'utf-8-sig\'))\n\ndef month_ord(s):\n    m=re.fullmatch(r\'(\\d{4})-(\\d\\d)\', str(s))\n    if not m: raise ValueError(\'bad month: \'+str(s))\n    y,n=map(int,m.groups());\n    if n not in range(1,13): raise ValueError(\'bad month\')\n    return y*12+n-1\n\ndef ymonth(n): return f\'{n//12:04d}-{n%12+1:02d}\'\ndef first_day(m): return str(m)+\'-01\'\ndef last_day(m):\n    o=month_ord(m)\n    return str(dt.date.fromisoformat(first_day(ymonth(o+1)))-dt.timedelta(days=1))\n\ndef quarter_end(q):\n    if not re.fullmatch(r\'\\d{4}Q[1-4]\', str(q)): raise ValueError(\'bad quarter\')\n    y=int(q[:4]); n=int(q[-1])*3\n    return last_day(f\'{y}-{n:02d}\')\n\ndef listing_date(value):\n    s=re.sub(r\'[^0-9]\', \'\', str(value or \'\'))\n    if len(s)==7: s=str(int(s[:3])+1911)+s[3:]\n    if len(s)!=8: return None\n    try:\n        d=dt.date(int(s[:4]),int(s[4:6]),int(s[6:8]));\n        if d<dt.date(1990,1,1) or d>dt.date(2026,10,24):return None\n        return d.isoformat()\n    except ValueError: return None\n\ndef parse_official_master(market: str, raw: bytes):\n    if market==\'twse\':\n        j=json.loads(raw)\n        if not isinstance(j,list): raise ValueError(\'TWSE master not list\')\n        rows=j\n    else:\n        rows=None\n        for enc in (\'utf-8-sig\',\'cp950\',\'big5\'):\n            try:\n                rows=list(csv.DictReader(io.StringIO(raw.decode(enc))));break\n            except UnicodeDecodeError: pass\n        if rows is None: raise ValueError(\'TPEx master encoding\')\n    if not rows or not any(\'公司代號\' in r for r in rows): raise ValueError(\'official roster schema\')\n    result={}\n    for r in rows:\n        symbol=str(r.get(\'公司代號\',\'\')).strip(); name=str(r.get(\'公司名稱\',\'\')).strip()\n        if not re.fullmatch(r\'[1-9][0-9]{3}\',symbol) or symbol.startswith(\'91\'): continue\n        if re.search(\'存託|TDR|特別股|受益證券|基金|權證|ETF|ETN\',name,re.I): continue\n        ld=None\n        for k,v in r.items():\n            if \'上市日期\' in str(k) or \'上櫃日期\' in str(k) or str(k).strip() in (\'上市日\',\'上櫃日\'):\n                ld=listing_date(v)\n                if ld: break\n        result[(market,symbol)]={\'name\':name,\'listing_date\':ld}\n    if not result: raise ValueError(\'official roster produced zero candidates\')\n    return result\n\ndef read_sources(source_dir: Path):\n    source_dir=Path(source_dir).resolve()\n    for n in REQUIRED:\n        if not (source_dir/n).is_file(): raise FileNotFoundError(f\'需要第20版盤點產物：{source_dir/n}\')\n    summary=json.loads((source_dir/REQUIRED[0]).read_text(\'utf-8\'))\n    src=summary.get(\'source\',{})\n    if src.get(\'release\')!=EXPECTED_RELEASE or src.get(\'current_sha256\')!=EXPECTED_CURRENT_SHA or src.get(\'manifest_sha256\')!=EXPECTED_MANIFEST_SHA:\n        raise ValueError(\'研究資料版本不一致；不對其他 release 任意補收\')\n    if summary.get(\'target_valid_monthends_per_PE_or_PB\')!=30 or summary.get(\'requests_executed\')!=0:\n        raise ValueError(\'第20版預查欄位不符\')\n    data={n:records(source_dir/n) for n in REQUIRED if n.endswith(\'.csv\')}\n    meta={\'source\':src,\'as_of_month\':summary[\'existing_last_month\'],\'existing_first_month\':summary[\'existing_first_month\'],\n          \'input_sha256\':{n:sha((source_dir/n).read_bytes()) for n in REQUIRED},\n          \'summary_snapshot\':{k:summary.get(k) for k in (\'ordinary_roster_candidates_not_PIT_verified\',\'stock_branch_pairs_below_30\')}}\n    return data,meta\n\ndef select_tasks(data, meta, official: dict, *, max_lookback_months=60):\n    """Fail closed on IPO-before-listing, nonpositive, and unsupported scope."""\n    asof=meta[\'as_of_month\']; first=ymonth(month_ord(asof)-max_lookback_months+1)\n    src01={ (r[\'market\'].lower().strip(),r[\'symbol\'].strip()):r for r in data[REQUIRED[1]] }\n    below={}\n    for r in data[REQUIRED[3]]:\n        key=(r[\'market\'].lower().strip(),r[\'symbol\'].strip(),r[\'branch\'].strip().lower())\n        if key[2] not in (\'pe\',\'pb\'): continue\n        if int(float(r[\'shortfall\']))>0: below[key]=r\n    gaps=defaultdict(set)\n    for r in data[REQUIRED[2]]:\n        key=(r[\'market\'].lower().strip(),r[\'symbol\'].strip(),r[\'branch\'].strip().lower())\n        if key in below and r.get(\'gap_reason\') in (\'MISSING_MONTH_ROW\',\'NULL_MULTIPLE\'):\n            gaps[key].add(r[\'month\'])\n    financial=defaultdict(set)\n    for r in data[REQUIRED[4]]:\n        market,symbol=r[\'market\'].lower().strip(),r[\'symbol\'].strip()\n        dataset={\'TaiwanStockFinancialStatements\':\'income\',\'TaiwanStockBalanceSheet\':\'balance\'}.get(r.get(\'dataset\'))\n        if dataset and (market,symbol) in src01 and str(src01[(market,symbol)].get(\'has_any_valuation_evidence\',\'\')).lower()==\'true\':\n            financial[(market,symbol,dataset)].update(q.strip() for q in str(r.get(\'missing_quarters\',\'\')).split(\',\') if re.fullmatch(r\'\\d{4}Q[1-4]\',q.strip()))\n    grouped=defaultdict(lambda: {\'branches\':set(),\'months\':set(),\'earlier\':False})\n    blocked=[]\n    for (market,symbol,branch),r in sorted(below.items()):\n        k=(market,symbol); ordinary=official.get(k)\n        base=src01.get(k)\n        if not ordinary or not base or str(base.get(\'has_any_valuation_evidence\',\'\')).lower()!=\'true\':\n            blocked.append({\'market\':market,\'symbol\':symbol,\'branch\':branch,\'reason\':\'NOT_CONFIRMED_CURRENT_ORDINARY_OR_NO_EXISTING_VALUATION\'});continue\n        listing=ordinary.get(\'listing_date\')\n        initial_month=base.get(\'first_observed_month\') or meta.get(\'existing_first_month\',\'\')\n        current_gaps=[]\n        for month in gaps.get((market,symbol,branch),set()):\n            if month_ord(month)<month_ord(first) or month_ord(month)>month_ord(asof): continue\n            # IPO unknown: only query months at/after existing documented valuation observations.\n            if listing and last_day(month)<listing: continue\n            if not listing and initial_month and month_ord(month)<month_ord(initial_month):continue\n            current_gaps.append(month)\n        state=grouped[k]\n        state[\'branches\'].add(branch)\n        state[\'months\'].update(current_gaps)\n        # Earlier-history is only eligible with authoritative current listing date available;\n        # not an assertion that PIT or historical ordinary-share classification is proven.\n        existing_first=meta.get(\'existing_first_month\',\'2023-10\')\n        if r.get(\'may_need_pre_existing_history_before\') and listing and listing<first_day(existing_first):\n            prior_months=[ymonth(n) for n in range(month_ord(first),month_ord(existing_first)) if last_day(ymonth(n))>=listing]\n            # Need only enough candidate months to fill shortfall + capped buffer. Latest first.\n            need=min(len(prior_months),int(float(r[\'shortfall\']))+4)\n            state[\'months\'].update(prior_months[-need:]);state[\'earlier\']=bool(need)\n        if not current_gaps and not state[\'earlier\']:\n            blocked.append({\'market\':market,\'symbol\':symbol,\'branch\':branch,\'reason\':\'NO_SAFE_MISSING_OR_NULL_MONTHS_OR_PRELISTING_HISTORY\'})\n    tasks=[]\n    for (market,symbol),r in sorted(grouped.items()):\n        if not r[\'months\']:continue\n        mn=min(r[\'months\']);mx=max(r[\'months\'])\n        tasks.append({\'kind\':\'valuation\',\'dataset\':DATASETS[\'valuation\'],\'market\':market,\'symbol\':symbol,\n                      \'start_date\':first_day(mn),\'end_date\':last_day(mx),\'target_months\':sorted(r[\'months\']),\n                      \'branches\':sorted(r[\'branches\']),\'listing_date\':official[(market,symbol)].get(\'listing_date\'),\n                      \'research_only\':True,\'reason\':\'MISSING_OR_NULL_MONTH_REVIEW\'})\n    # Only financial quarters relevant to the observed / requested valuation study interval;\n    # don\'t infer missing data from absent published_at; don\'t request IPO-prelisting quarters.\n    for (market,symbol,kind),qset in sorted(financial.items()):\n        if (market,symbol) not in src01 or (market,symbol) not in official:continue\n        base=src01[(market,symbol)]\n        listing=official[(market,symbol)].get(\'listing_date\')\n        # Without IPO date, conservative earliest observed valuation month as lower bound.\n        visible=base.get(\'first_observed_month\') or meta.get(\'existing_first_month\',\'\')\n        lower=(min(grouped[(market,symbol)][\'months\']) if (market,symbol) in grouped and grouped[(market,symbol)][\'months\'] else visible)\n        start=ymonth(max(month_ord(first),month_ord(lower)-12)) # financial warmup\n        valid=[]\n        for q in sorted(qset):\n            d=quarter_end(q)\n            if d>last_day(asof): continue\n            if month_ord(d[:7])<month_ord(start):continue\n            if listing and d<listing:continue\n            if not listing and visible and month_ord(d[:7])<month_ord(visible):continue\n            valid.append(q)\n        if not valid:continue\n        tasks.append({\'kind\':kind,\'dataset\':DATASETS[kind],\'market\':market,\'symbol\':symbol,\n                      \'start_date\':quarter_end(min(valid)),\'end_date\':quarter_end(max(valid)),\n                      \'target_quarters\':valid,\'listing_date\':listing,\n                      \'research_only\':True,\'reason\':\'WHOLE_STATEMENT_QUARTER_ABSENT_SOURCE_CHECK\'})\n    return sorted(tasks,key=lambda r:(r[\'symbol\'],r[\'dataset\'],r[\'start_date\'])),blocked\n\ndef task_id(task):\n    x={k:task[k] for k in (\'dataset\',\'symbol\',\'start_date\',\'end_date\')}\n    return sha(json.dumps(x,sort_keys=True).encode())[:20]\n\ndef verify_receipt(root: Path, task):\n    tid=task_id(task);receipt=root/\'receipts\'/f\'{tid}.json\';raw=root/\'raw\'/f\'{tid}.json\'\n    if not receipt.is_file() or not raw.is_file():return None\n    try:\n        r=json.loads(receipt.read_text(\'utf-8\'))\n        if r[\'task_id\']!=tid or r[\'dataset\']!=task[\'dataset\'] or r[\'symbol\']!=task[\'symbol\'] or r[\'raw_sha256\']!=sha(raw.read_bytes()):return None\n        if r.get(\'status\') not in (\'SUCCESS\',\'EMPTY_SOURCE_RESPONSE\'):return None\n        j=json.loads(raw.read_bytes())\n        if str(j.get(\'status\'))!=\'200\' or not isinstance(j.get(\'data\'),list):return None\n        return r\n    except (ValueError,KeyError):return None\n\ndef response_kind(status_code, payload):\n    if status_code in (401,403) or str(payload.get(\'status\')) in (\'401\',\'403\'):return \'ACCESS_DENIED\'\n    if status_code in (402,429) or str(payload.get(\'status\')) in (\'402\',\'429\'):return \'QUOTA_PAUSED\'\n    if status_code>=500:return \'RETRYABLE_HTTP\'\n    if status_code!=200:return \'ACCESS_OR_HTTP_ERROR\'\n    if not isinstance(payload,dict) or str(payload.get(\'status\'))!=\'200\' or not isinstance(payload.get(\'data\'),list):return \'SCHEMA_OR_SOURCE_ERROR\'\n    return \'EMPTY_SOURCE_RESPONSE\' if not payload[\'data\'] else \'SUCCESS\'\n\ndef safe_finmind_response(body:bytes, task):\n    try:j=json.loads(body)\n    except (ValueError,UnicodeDecodeError):return \'SCHEMA_OR_SOURCE_ERROR\',0\n    if not isinstance(j,dict):return \'SCHEMA_OR_SOURCE_ERROR\',0\n    if j.get(\'status\') not in (200,\'200\'):\n        return response_kind(200,{\'status\':int(j.get(\'status\',0)) if str(j.get(\'status\',\'\')).isdigit() else 0}),0\n    rows=j.get(\'data\')\n    if not isinstance(rows,list):return \'SCHEMA_OR_SOURCE_ERROR\',0\n    for item in rows:\n        if not isinstance(item,dict) or str(item.get(\'stock_id\',\'\')).strip()!=task[\'symbol\']:\n            return \'SCHEMA_OR_SOURCE_ERROR\',0\n        if task[\'kind\']==\'valuation\':\n            date=item.get(\'date\')\n            if not date or not str(date)[:10].startswith((\'19\',\'20\')):\n                return \'SCHEMA_OR_SOURCE_ERROR\',0\n            if not (\'PER\' in item and \'PBR\' in item):return \'SCHEMA_OR_SOURCE_ERROR\',0\n        else:\n            if not {\'date\',\'type\',\'value\'}.issubset(item):return \'SCHEMA_OR_SOURCE_ERROR\',0\n    return (\'SUCCESS\' if rows else \'EMPTY_SOURCE_RESPONSE\'),len(rows)\n\ndef persist_response(root, task, response, status, count):\n    tid=task_id(task);raw=root/\'raw\'/f\'{tid}.json\';receipt=root/\'receipts\'/f\'{tid}.json\'\n    atomic_bytes(raw,response)\n    r={\'task_id\':tid,\'status\':status,\'dataset\':task[\'dataset\'],\'kind\':task[\'kind\'],\n       \'market\':task[\'market\'],\'symbol\':task[\'symbol\'],\'start_date\':task[\'start_date\'],\n       \'end_date\':task[\'end_date\'],\'retrieved_at_utc\':utc(),\'raw_sha256\':sha(response),\n       \'raw_bytes\':len(response),\'rows\':count,\'historical_pit_verified\':False,\n       \'publication_allowed\':False,\'source_authorization\':\'REVIEW_REQUIRED\',\n       \'month_end_official_verified\':False,\'request_auth\':\'BEARER_HEADER_NOT_PERSISTED\'}\n    atomic_json(receipt,r)\n    return r\n\ndef run_fetch(root, tasks, *, token, max_calls_per_run=25, sleep_seconds=8.0,\n              session=None, sleeper=time.sleep, retry_once=False):\n    import requests\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    if not token:raise RuntimeError(\'FinMind Token 未設定；無法發送補收請求\')\n    if not (1<=max_calls_per_run<=500):raise ValueError(\'MAX_CALLS_PER_RUN 必須在 1..500\')\n    if sleep_seconds < 7.2:raise ValueError(\'避免超出免費額度，請將請求間隔維持 >=7.2 秒\')\n    ses=session or requests.Session()\n    actual=0;statuses=[];quota_stopped=False;last_call=None\n    for task in tasks:\n        cache=verify_receipt(root,task)\n        if cache:\n            statuses.append({\'task_id\':task_id(task),\'status\':\'CACHED_\'+cache[\'status\']});continue\n        if actual>=max_calls_per_run or quota_stopped: break\n        # Avoid token in query / request URL. Do not log exception with request path.\n        params={\'dataset\':task[\'dataset\'],\'data_id\':task[\'symbol\'],\n                \'start_date\':task[\'start_date\'],\'end_date\':task[\'end_date\']}\n        headers={\'Authorization\':\'Bearer \'+token,\'User-Agent\':\'Janus-Valuation-04-Research-Backfill/2.1\'}\n        if last_call is not None:sleeper(max(0,sleep_seconds-(time.monotonic()-last_call)))\n        attempts=2 if retry_once else 1\n        for attempt in range(attempts):\n            if actual>=max_calls_per_run:break\n            last_call=time.monotonic();actual+=1\n            try:\n                r=ses.get(BASE_URL,params=params,headers=headers,timeout=(12,60))\n                payload=r.content\n                try: parsed=json.loads(payload)\n                except (ValueError,UnicodeDecodeError):parsed={}\n                if isinstance(parsed,dict) and str(parsed.get(\'status\')) in (\'402\',\'429\'):\n                    kind=\'QUOTA_PAUSED\'\n                else:\n                    kind=response_kind(r.status_code, parsed if isinstance(parsed,dict) else {})\n                if kind in (\'SUCCESS\',\'EMPTY_SOURCE_RESPONSE\'):\n                    kind,n=safe_finmind_response(payload,task)\n                    if kind in (\'SUCCESS\',\'EMPTY_SOURCE_RESPONSE\'):\n                        persist_response(root,task,payload,kind,n)\n                if kind==\'QUOTA_PAUSED\':quota_stopped=True\n            except requests.RequestException:\n                kind=\'RETRYABLE_NETWORK\' # never print potentially token-bearing URL\n            statuses.append({\'task_id\':task_id(task),\'dataset\':task[\'dataset\'],\n                             \'symbol\':task[\'symbol\'],\'status\':kind,\'attempt\':attempt+1})\n            atomic_json(root/\'last_progress.json\',{\'updated_at\':utc(),\'http_calls_this_run\':actual,\n                      \'last_symbol\':task[\'symbol\'],\'last_dataset\':task[\'dataset\'],\n                      \'last_status\':kind,\'quota_paused\':quota_stopped,\n                      \'publication_allowed\':False})\n            if kind not in (\'RETRYABLE_HTTP\',\'RETRYABLE_NETWORK\') or attempt==attempts-1:break\n            sleeper(max(15.0,sleep_seconds));last_call=time.monotonic()\n        if quota_stopped or actual>=max_calls_per_run: break\n    return {\'actual_http_calls_this_run\':actual,\'cached_verified\':sum(1 for s in statuses if s[\'status\'].startswith(\'CACHED_\')),\n            \'success_or_empty_this_run\':sum(1 for s in statuses if s[\'status\'] in (\'SUCCESS\',\'EMPTY_SOURCE_RESPONSE\')),\n            \'quota_paused\':quota_stopped,\'attempts\':statuses[-60:]}\n\ndef generate_reports(root, tasks, base_data, *, run_summary=None):\n    receipts=[verify_receipt(Path(root),t) for t in tasks]\n    statuses=[{\'task_id\':task_id(t),\'symbol\':t[\'symbol\'],\'dataset\':t[\'dataset\'],\n              \'kind\':t[\'kind\'],\'start_date\':t[\'start_date\'],\'end_date\':t[\'end_date\'],\n              \'status\':(\'FINISHED_\'+r[\'status\'] if r else \'PENDING_OR_ERROR\'),\n              \'rows\':(r.get(\'rows\',0) if r else 0)} for t,r in zip(tasks,receipts)]\n    atomic_csv(root/\'01_task_status.csv\',statuses,[\'task_id\',\'symbol\',\'dataset\',\'kind\',\'start_date\',\'end_date\',\'status\',\'rows\'])\n    # Only candidate positive PE/PB, not verified true exchange month-end or safe-to-train PIT.\n    base={(r[\'market\'].lower().strip(),r[\'symbol\'].strip()):r for r in base_data[REQUIRED[1]]}\n    more=defaultdict(lambda:{\'pe\':set(),\'pb\':set()})\n    allowed={(r[\'market\'].lower().strip(),r[\'symbol\'].strip(),r[\'branch\'].strip().lower(),r[\'month\'])\n      for r in base_data[REQUIRED[2]] if r.get(\'gap_reason\') in (\'MISSING_MONTH_ROW\',\'NULL_MULTIPLE\')}\n    val_rows=[];financial_rows=[]\n    for t,r in zip(tasks,receipts):\n        if not r or r.get(\'status\')!=\'SUCCESS\':continue\n        blob=json.loads((root/\'raw\'/(task_id(t)+\'.json\')).read_bytes())[\'data\']\n        if t[\'kind\']==\'valuation\':\n            by_month=defaultdict(list)\n            for x in blob:\n                date=str(x.get(\'date\',\'\'))[:10]\n                try:d=dt.date.fromisoformat(date)\n                except ValueError: continue\n                if not (t[\'start_date\']<=date<=t[\'end_date\']):continue\n                if t.get(\'listing_date\') and date<t[\'listing_date\']:continue\n                by_month[date[:7]].append((date,x))\n            for month,pairs in sorted(by_month.items()):\n                # use last actual observation from this HTTP response, no synthetic monthends\n                date,x=max(pairs,key=lambda y:y[0])\n                for branch,col in ((\'pe\',\'PER\'),(\'pb\',\'PBR\')):\n                    v=x.get(col)\n                    try:v=float(v)\n                    except (ValueError,TypeError):v=math.nan\n                    valid=math.isfinite(v) and v>0\n                    candidate=(t[\'market\'],t[\'symbol\'],branch,month) in allowed or (month in t.get(\'target_months\',[]) and month<base[(t[\'market\'],t[\'symbol\'])].get(\'first_observed_month\',\'9999-99\'))\n                    if valid and candidate:more[(t[\'market\'],t[\'symbol\'])][branch].add(month)\n                    val_rows.append({\'market\':t[\'market\'],\'symbol\':t[\'symbol\'],\'month\':month,\'observed_date\':date,\n                                     \'branch\':branch,\'multiple\':v if math.isfinite(v) else \'\',\n                                     \'positive\':valid,\'candidate_month\':candidate,\'source_raw_sha256\':r[\'raw_sha256\'],\n                                     \'status\':\'RESEARCH_CANDIDATE_PIT_AND_MONTHEND_UNVERIFIED\'})\n        else:\n            per_q=defaultdict(int)\n            for row in blob:\n                date=str(row.get(\'date\',\'\'))[:10]\n                if date:\n                    per_q[date[:7]]+=1\n            for q in t[\'target_quarters\']:\n                financial_rows.append({\'market\':t[\'market\'],\'symbol\':t[\'symbol\'],\'dataset\':t[\'dataset\'],\n                                       \'fiscal_quarter\':q,\'source_rows_with_same_quarter_end\':per_q.get(quarter_end(q)[:7],0),\n                                       \'status\':\'RAW_RESPONSE_ONLY_SCOPE_UNIT_PIT_UNVERIFIED\',\'source_raw_sha256\':r[\'raw_sha256\']})\n    atomic_csv(root/\'02_valuation_recovered_month_candidates.csv\',val_rows,\n       [\'market\',\'symbol\',\'month\',\'observed_date\',\'branch\',\'multiple\',\'positive\',\'candidate_month\',\'source_raw_sha256\',\'status\'])\n    atomic_csv(root/\'03_financial_source_check.csv\',financial_rows,\n       [\'market\',\'symbol\',\'dataset\',\'fiscal_quarter\',\'source_rows_with_same_quarter_end\',\'status\',\'source_raw_sha256\'])\n    scores=[]\n    for key,v in sorted(base.items()):\n        for branch in (\'pe\',\'pb\'):\n            n=int(float(v.get(f\'{branch}_positive_months\') or 0));additional=len(more[key][branch]);\n            scores.append({\'market\':key[0],\'symbol\':key[1],\'branch\':branch,\'existing_positive_months\':n,\n                           \'additional_positive_month_candidates\':additional,\n                           \'possible_positive_months_not_pit_verified\':n+additional,\n                           \'precheck_30_positive_months\':n+additional>=30,\n                           \'model_ready\':False,\'publication_allowed\':False})\n    atomic_csv(root/\'04_postfetch_positive_coverage_precheck.csv\',scores,\n       [\'market\',\'symbol\',\'branch\',\'existing_positive_months\',\'additional_positive_month_candidates\',\n        \'possible_positive_months_not_pit_verified\',\'precheck_30_positive_months\',\'model_ready\',\'publication_allowed\'])\n    summary={\'status\':\'FETCH_ATTEMPTED_NOT_MODEL_READY\',\'tasks_total\':len(tasks),\n      \'tasks_success\':sum(1 for r in receipts if r and r.get(\'status\')==\'SUCCESS\'),\n      \'tasks_empty\':sum(1 for r in receipts if r and r.get(\'status\')==\'EMPTY_SOURCE_RESPONSE\'),\n      \'tasks_pending\':sum(1 for r in receipts if not r),\n      \'candidate_positive_months_added\':sum(len(x[\'pe\'])+len(x[\'pb\']) for x in more.values()),\n      \'run_stats\':run_summary or {},\'publication_allowed\':False,\n      \'limitations\':[\'FinMind daily last observed is not certified exchange month-end\',\'positive multiples do not certify PIT, units or financial comparability\',\'historical source versions are not reconstructed\'],\n      \'updated_utc\':utc()}\n    atomic_json(root/\'00_run_summary.json\',summary)\n    return summary\n'
_base = types.ModuleType('valuation_min30_real_backfill')
sys.modules[_base.__name__] = _base
exec(compile(_base_code,'valuation_min30_real_backfill.py','exec'),_base.__dict__)
print('v21 core source SHA256',hashlib.sha256(_base_code.encode()).hexdigest())

In [ ]:
# 3. V22 歷史回溯新選擇器 + 收據讀回 + immutable plan
_v22_code = '"""V22: evidence-gated historical Valuation research backfill, read-only old receipts.\nNo canonical writes; FinMind fetch only via an explicit bounded call.\n"""\nfrom __future__ import annotations\nimport csv\nimport datetime as dt\nimport hashlib\nimport json\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\nimport valuation_min30_real_backfill as base\n\nVERSION=\'VAL-MIN30-HIST-V22-20261024\'\n# Minimum positive observation months to prioritize a historical query, not model eligibility.\nMIN_OBSERVED_POSITIVE_FOR_FETCH=12\nMAX_HIST_MONTHS_PER_TASK=24\n\n\ndef digest_json(x):\n    return base.sha(json.dumps(x,ensure_ascii=False,sort_keys=True, separators=(\',\',\':\')).encode())\n\n\ndef positive(v):\n    try:\n        x=float(v)\n        return math.isfinite(x) and x>0\n    except (ValueError,TypeError):\n        return False\n\n\ndef inspect_legacy(legacy_root:Path):\n    """Never modifies v21. SHA-check every completed receipt before inspecting its raw."""\n    legacy_root=Path(legacy_root)\n    out={\'receipts_valid\':0,\'receipts_invalid\':0,\'raw_positive_months\':0,\n         \'raw_nonpositive_months\':0,\'validated\':[],\'positive_by_symbol_branch\':defaultdict(set),\n         \'nonpositive_by_symbol_branch\':defaultdict(set)}\n    if not (legacy_root/\'receipts\').is_dir():return out\n    for file in sorted((legacy_root/\'receipts\').glob(\'*.json\')):\n        try:\n            rec=json.loads(file.read_text(\'utf-8\'))\n            if rec.get(\'dataset\')!=base.DATASETS[\'valuation\'] or rec.get(\'status\') not in (\'SUCCESS\',\'EMPTY_SOURCE_RESPONSE\'):\n                continue\n            raw=legacy_root/\'raw\'/file.name\n            if rec.get(\'task_id\')!=file.stem or not raw.is_file() or rec.get(\'raw_sha256\')!=base.sha(raw.read_bytes()):\n                out[\'receipts_invalid\']+=1;continue\n            payload=json.loads(raw.read_bytes())\n            if str(payload.get(\'status\'))!=\'200\' or not isinstance(payload.get(\'data\'),list):\n                out[\'receipts_invalid\']+=1;continue\n            sym=str(rec[\'symbol\']); monthly=defaultdict(list)\n            for row in payload[\'data\']:\n                if str(row.get(\'stock_id\'))!=sym:\n                    raise ValueError(\'v21 mismatched stock_id\')\n                date=dt.date.fromisoformat(row[\'date\'])\n                if not rec[\'start_date\']<=date.isoformat()<=rec[\'end_date\']:continue\n                monthly[date.strftime(\'%Y-%m\')].append((date,row))\n            for month,rows in monthly.items():\n                date,last=max(rows,key=lambda x:x[0])\n                for branch,col in ((\'pe\',\'PER\'),(\'pb\',\'PBR\')):\n                    key=(str(rec[\'market\']).lower(),sym,branch)\n                    dest=\'positive_by_symbol_branch\' if positive(last.get(col)) else \'nonpositive_by_symbol_branch\'\n                    out[dest][key].add(month)\n            out[\'receipts_valid\']+=1\n            out[\'validated\'].append({\'task_id\':file.stem,\'symbol\':sym,\'raw_sha256\':rec[\'raw_sha256\'],\n                                      \'start_date\':rec[\'start_date\'],\'end_date\':rec[\'end_date\']})\n        except (ValueError,KeyError,TypeError,UnicodeError):\n            out[\'receipts_invalid\']+=1\n    out[\'raw_positive_months\']=sum(map(len,out[\'positive_by_symbol_branch\'].values()))\n    out[\'raw_nonpositive_months\']=sum(map(len,out[\'nonpositive_by_symbol_branch\'].values()))\n    return out\n\n\ndef select_v22(data:dict, meta:dict, official:dict, legacy:dict, *,\n               months_back:int=60, min_positive:int=MIN_OBSERVED_POSITIVE_FOR_FETCH,\n               historical_cap:int=MAX_HIST_MONTHS_PER_TASK):\n    """Select only pre-release valuation dates; never retry a known PE=0 in-place.\n\n    Current official listing date is an upper-bound eligibility guard only; no historical PIT claim.\n    """\n    if months_back<36 or months_back>120:raise ValueError(\'months_back must be 36..120\')\n    if historical_cap<1 or historical_cap>48:raise ValueError(\'cap must be 1..48\')\n    end=base.month_ord(meta[\'as_of_month\'])\n    existing_start=base.month_ord(meta[\'existing_first_month\'])\n    earliest=end-months_back+1\n    if existing_start<=earliest:raise ValueError(\'no pre-release history inside lookback\')\n    universe={(r[\'market\'].strip().lower(),r[\'symbol\'].strip()):r for r in data[base.REQUIRED[1]]}\n    grouped=defaultdict(lambda:{\'months\':set(),\'branches\':set(),\'branch_detail\':[]})\n    holds=[]\n    pairs=[]\n    for r in data[base.REQUIRED[3]]:\n        market,symbol,branch=r[\'market\'].strip().lower(),r[\'symbol\'].strip(),r[\'branch\'].lower().strip()\n        if branch not in (\'pe\',\'pb\'):continue\n        short=int(float(r[\'shortfall\']))\n        if short<=0:continue\n        baseline=universe.get((market,symbol)); ordinary=official.get((market,symbol))\n        n=int(float((baseline or {}).get(branch+\'_positive_months\') or 0))\n        k=(market,symbol,branch)\n        decision={\'market\':market,\'symbol\':symbol,\'branch\':branch,\'baseline_positive_months\':n,\n                  \'shortfall\':short,\'status\':\'\',\'historical_months\':0,\'reason\':\'\'}\n        if not baseline or str(baseline.get(\'has_any_valuation_evidence\')).lower()!=\'true\' or not ordinary:\n            decision.update(status=\'HOLD\',reason=\'UNVERIFIED_ORDINARY_OR_NO_VALUATION\')\n        elif not ordinary.get(\'listing_date\'):\n            decision.update(status=\'HOLD\',reason=\'UNKNOWN_LISTING_DATE_FOR_PRE_2023\')\n        elif n<min_positive:\n            decision.update(status=\'HOLD\',reason=\'LOW_POSITIVE_HISTORY_YIELD_AUDIT_FIRST\')\n        else:\n            listing=ordinary[\'listing_date\']\n            historic=[base.ymonth(i) for i in range(earliest,existing_start)\n                    if base.last_day(base.ymonth(i))>=listing]\n            # For low probability PE=0 stretches, cap to shortfall+4 and then measure.\n            need=min(len(historic),short+4,historical_cap)\n            selected=historic[-need:] if need>0 else []\n            if not selected:\n                decision.update(status=\'HOLD\',reason=\'NO_PRE_2023_POST_LISTING_MONTHS\')\n            else:\n                decision.update(status=\'FETCH_CANDIDATE\',reason=\'EARLIER_UNOBSERVED_MONTHS\',historical_months=len(selected))\n                group=grouped[(market,symbol)]\n                group[\'months\'].update(selected)\n                group[\'branches\'].add(branch)\n                group[\'branch_detail\'].append({\'branch\':branch,\'baseline\':n,\'shortfall\':short,\'historical_months\':len(selected)})\n        pairs.append(decision)\n        if decision[\'status\']==\'HOLD\':holds.append(decision)\n    tasks=[]\n    for (market,symbol),g in sorted(grouped.items()):\n        months=sorted(g[\'months\']);listing=official[(market,symbol)][\'listing_date\']\n        if months[-1]>=meta[\'existing_first_month\']:raise ValueError(\'v22 would re-fetch v21 range\')\n        tasks.append({\'dataset\':base.DATASETS[\'valuation\'],\'kind\':\'valuation\', \'market\':market,\'symbol\':symbol,\n                \'start_date\':base.first_day(months[0]), \'end_date\':base.last_day(months[-1]),\n                \'target_months\':months,\'branches\':sorted(g[\'branches\']),\n                \'listing_date\':listing,\'reason\':\'PRE_RELEASE_MIN30_TARGETED_HISTORY\',\n                \'branch_detail\':sorted(g[\'branch_detail\'],key=lambda x:x[\'branch\']),\n                \'publication_allowed\':False,\'research_only\':True,\'plan_version\':VERSION})\n    # Highest near-target probability first; avoid waste on chronic PER=0.\n    tasks.sort(key=lambda t:(-max(x[\'baseline\'] for x in t[\'branch_detail\']),\n                             min(x[\'shortfall\'] for x in t[\'branch_detail\']),t[\'market\'],t[\'symbol\']))\n    return tasks,holds,pairs\n\n\ndef fingerprint(tasks, meta, official, legacy):\n    official_fingerprint=[{\'market\':k[0],\'symbol\':k[1],\'listing_date\':v.get(\'listing_date\')}\n                          for k,v in sorted(official.items())]\n    return digest_json({\'version\':VERSION,\'tasks\':tasks,\'source\':meta[\'input_sha256\'],\n                        \'source_release\':meta[\'source\'],\'master\':official_fingerprint})\n\n\ndef freeze(root:Path,tasks,meta,official,legacy,holds,pairs):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    token=fingerprint(tasks,meta,official,legacy)\n    path=root/\'frozen_plan_v22.json\'\n    if path.exists():\n        j=json.loads(path.read_text(\'utf-8\'))\n        if j.get(\'fingerprint\')!=token:raise RuntimeError(\'immutable v22 plan drift; use a new versioned folder\')\n        return j\n    # Existing v22 receipts without a plan would create an unsafe partial resume.\n    if (root/\'receipts\').exists() and any((root/\'receipts\').glob(\'*.json\')):\n        raise RuntimeError(\'v22 receipts exist without a frozen plan\')\n    plan={\'version\':VERSION,\'fingerprint\':token,\'tasks\':tasks,\n          \'input_hashes\':meta[\'input_sha256\'],\'source\':meta[\'source\'],\n          \'legacy_sha_verified\':legacy[\'receipts_valid\'], \'legacy_sha_invalid\':legacy[\'receipts_invalid\'],\n          \'hold_pairs\':len(holds),\'pairs_reviewed\':len(pairs),\'publication_allowed\':False}\n    base.atomic_json(path,plan)\n    base.atomic_csv(root/\'v22_pairs.csv\',pairs,[\'market\',\'symbol\',\'branch\',\'baseline_positive_months\',\n       \'shortfall\',\'status\',\'historical_months\',\'reason\'])\n    base.atomic_csv(root/\'v22_tasks.csv\',[\n        {\'task_id\':base.task_id(t),\'symbol\':t[\'symbol\'],\'market\':t[\'market\'],\'start_date\':t[\'start_date\'],\n         \'end_date\':t[\'end_date\'],\'target_months\':len(t[\'target_months\']),\n         \'branches\':\',\'.join(t[\'branches\'])} for t in tasks],\n         [\'task_id\',\'symbol\',\'market\',\'start_date\',\'end_date\',\'target_months\',\'branches\'])\n    base.atomic_json(root/\'v21_reuse_index.json\',{\'status\':\'SHA_VERIFIED_READ_ONLY\',\'count_valid\':legacy[\'receipts_valid\'],\n        \'count_invalid\':legacy[\'receipts_invalid\'],\'prior_raw_positive_months\':legacy[\'raw_positive_months\'],\n        \'prior_raw_nonpositive_months\':legacy[\'raw_nonpositive_months\'], \'receipts\':legacy[\'validated\'],\n        \'note\':\'V21 receipts are preserved and are not copied into V22 output; no current-range refetch.\'})\n    return plan\n\n\ndef yield_report(root:Path,tasks:list,existing:dict,legacy:dict,run_stats=None):\n    """Positive month-end candidates only in requested prehistory; never certifies PIT or official closing date."""\n    gained=defaultdict(set)\n    taskrows=[]\n    for task in tasks:\n        r=base.verify_receipt(Path(root),task)\n        row={\'task_id\':base.task_id(task),\'symbol\':task[\'symbol\'],\'status\':\'PENDING_OR_ERROR\' if r is None else r[\'status\'],\n             \'rows\':0 if r is None else r[\'rows\'],\'candidate_positive_months\':0}\n        if r and r[\'status\']==\'SUCCESS\':\n            payload=json.loads((Path(root)/\'raw\'/(base.task_id(task)+\'.json\')).read_bytes())\n            by_month=defaultdict(list)\n            for x in payload[\'data\']:\n                day=str(x.get(\'date\',\'\'))[:10];m=day[:7]\n                if m not in task[\'target_months\'] or day<task[\'listing_date\'] or not task[\'start_date\']<=day<=task[\'end_date\']:continue\n                by_month[m].append(x)\n            for m,rows in by_month.items():\n                item=max(rows,key=lambda x:x[\'date\'])\n                for branch in task[\'branches\']:\n                    if positive(item.get(\'PER\' if branch==\'pe\' else \'PBR\')):\n                        gained[(task[\'market\'],task[\'symbol\'],branch)].add(m)\n            row[\'candidate_positive_months\']=sum(len(gained[(task[\'market\'],task[\'symbol\'],b)]) for b in task[\'branches\'])\n        taskrows.append(row)\n    base.atomic_csv(Path(root)/\'v22_task_status.csv\',taskrows,[\'task_id\',\'symbol\',\'status\',\'rows\',\'candidate_positive_months\'])\n    gain_rows=[]\n    for task in tasks:\n        for q in task[\'branch_detail\']:\n            key=(task[\'market\'],task[\'symbol\'],q[\'branch\'])\n            n=len(gained[key])\n            gain_rows.append({\'market\':key[0],\'symbol\':key[1],\'branch\':key[2],\n                \'existing_positive\':q[\'baseline\'],\'new_candidate_positive_months\':n,\n                \'combined_positive_precheck\':q[\'baseline\']+n, \'precheck_at_least_30\':q[\'baseline\']+n>=30,\n                \'model_ready\':False,\'publication_allowed\':False})\n    base.atomic_csv(Path(root)/\'v22_valid_month_gain.csv\',gain_rows,\n       [\'market\',\'symbol\',\'branch\',\'existing_positive\',\'new_candidate_positive_months\',\'combined_positive_precheck\',\'precheck_at_least_30\',\'model_ready\',\'publication_allowed\'])\n    positive_count=sum(len(x) for x in gained.values())\n    succeeded=sum(1 for r in taskrows if r[\'status\']==\'SUCCESS\')\n    overall={\'version\':VERSION,\'status\':\'RESEARCH_PRECHECK_NOT_MODEL_READY\',\'plan_tasks\':len(tasks),\n        \'tasks_success\':succeeded,\'tasks_pending\':sum(1 for r in taskrows if r[\'status\']==\'PENDING_OR_ERROR\'),\n        \'new_positive_month_candidate_count\':positive_count,\'legacy_receipts_sha_verified\':legacy[\'receipts_valid\'],\n        \'legacy_receipts_sha_invalid\':legacy[\'receipts_invalid\'],\'run\':run_stats or {},\n        \'publication_allowed\':False,\'source_pit_verified\':False,\n        \'recommendation\':(\'CONTINUE_BOUNDED\' if positive_count>0 else \'CHECK_FIRST_BATCH_YIELD_BEFORE_EXPANDING\'),\n        \'updated_at_utc\':base.utc()}\n    base.atomic_json(Path(root)/\'00_run_summary_v22.json\',overall)\n    return overall\n'
_v22 = types.ModuleType('valuation_min30_v22')
sys.modules[_v22.__name__] = _v22
exec(compile(_v22_code,'valuation_min30_v22.py','exec'),_v22.__dict__)
print('v22 version',_v22.VERSION,'SHA256',hashlib.sha256(_v22_code.encode()).hexdigest())

In [ ]:
# 4. 讀第20版真實清冊、驗證第21版原始收據 SHA，取得/固定上市名冊，生成版本化 v22 任務
D,M=_base.read_sources(SOURCE)
LEGACY_AUDIT=_v22.inspect_legacy(LEGACY)
print('v21已校驗receipt',LEGACY_AUDIT['receipts_valid'],'SHA/結構不符',LEGACY_AUDIT['receipts_invalid'])
# 務必先讀保存的V22主檔；不可每次resume以變動的今日名冊悄悄改動plan。
DEST.mkdir(parents=True,exist_ok=True)
master_path=DEST/'official_master_v22.json'
if master_path.exists():
    snap=json.loads(master_path.read_text('utf-8'))
    assert snap.get('version')==_v22.VERSION
    roster=snap['roster'];assert _v22.digest_json(roster)==snap['roster_sha256']
    OFFICIAL={tuple(key.split('|',1)):entry for key,entry in roster.items()}
else:
    OFFICIAL={}; evidence=[]
    for market,url in _base.OFFICIAL.items():
        resp=requests.get(url,timeout=(12,45),headers={'User-Agent':'Janus-Valuation-Research/2.2'})
        resp.raise_for_status()
        parsed=_base.parse_official_master(market,resp.content)
        OFFICIAL.update(parsed)
        evidence.append({'market':market,'url':url,'observed_at_utc':_base.utc(),
             'sha256':_base.sha(resp.content),'ordinary_candidate_count':len(parsed)})
    assert len(OFFICIAL)>1500,'官方名冊候選過少，停止'
    roster={a+'|'+b:entry for (a,b),entry in sorted(OFFICIAL.items())}
    _base.atomic_json(master_path,{'version':_v22.VERSION,'roster':roster,
             'roster_sha256':_v22.digest_json(roster),'evidence':evidence,
             'warning':'Today roster != historical PIT identity'})
TASKS,HOLDS,PAIRS=_v22.select_v22(D,M,OFFICIAL,LEGACY_AUDIT,
            months_back=MAX_LOOKBACK_MONTHS,min_positive=MIN_EXISTING_POSITIVE)
PLAN=_v22.freeze(DEST,TASKS,M,OFFICIAL,LEGACY_AUDIT,HOLDS,PAIRS)
assert PLAN['fingerprint']==_v22.fingerprint(TASKS,M,OFFICIAL,LEGACY_AUDIT)
assert all(t['end_date']<M['existing_first_month']+'-01' for t in TASKS)
print('新的早期估值任務',len(TASKS),'分支HOLD',len(HOLDS),
      '最早開始',min((t['start_date'] for t in TASKS),default=None))
print('Plan Fingerprint',PLAN['fingerprint'])

In [ ]:
# 5. 使用獨立v22凍結計畫執行最多5次HTTP（多次Run all即可resume）
# 如果已試抓過 >=5 筆，且候選新增有效正倍數仍為 0，先停止而不是浪費免費額度。
prior_path=DEST/'00_run_summary_v22.json'
if prior_path.exists():
    prior=json.loads(prior_path.read_text('utf-8'))
    if prior.get('tasks_success',0)>=5 and prior.get('new_positive_month_candidate_count',0)==0:
        raise RuntimeError('上一批實抓至少5筆仍無有效正月末，暫停擴大，先檢查回應/策略')
if not TASKS:
    print('本次沒有符合安全上市日及歷史PE/PB有效條件的任務；請檢查 v22_pairs.csv')
else:
    locked=DEST/'_ACTIVE_FETCH_LOCK'
    if locked.exists():
        raise RuntimeError('v22執行鎖仍存在；確認沒有其它Colab在跑後才人工處理')
    locked.mkdir()
    try:
        if RUN_FETCH:
            token=getpass.getpass('FinMind免費帳號Token（不落盤）: ').strip()
            if not token:raise RuntimeError('Token為空；不發請求')
            results=_base.run_fetch(DEST,TASKS,token=token,max_calls_per_run=MAX_CALLS_PER_RUN,
                       sleep_seconds=SLEEP_SECONDS,retry_once=RETRY_ONCE)
            del token
        else:
            results={'actual_http_calls_this_run':0,'status':'DRY_RUN_NO_FETCH'}
        summary=_v22.yield_report(DEST,TASKS,D,LEGACY_AUDIT,run_stats=results)
        print(json.dumps(summary,ensure_ascii=False,indent=2))
        print('這裡的新增月份僅研究候選，不代表公司行動／PIT／官方月末已核准。')
    finally:
        locked.rmdir()

### 續跑與停止條件

- 重新執行本 Notebook 會從同一個 `frozen_plan_v22.json`、收據 SHA 驗證後續跑；**不會修改 v21**。每輪最多 5 次 HTTP（明確自行修改上限後可到25）；配額、401/403、錯誤會依原引擎停止或保留 pending。
- 如果頭5筆取得資料但**對分支的新增有效正倍數月份為 0**，下一輪會自動擋住，請先核對資料供應及口徑。
- 原資料 `PER<=0`／`PBR<=0` 仍可能是真實狀態；此版只查既有研究資料最早月份以前、上市以後的 **新月份**。
- 當前名冊及來源歷史財報 PIT、公告時間等未完整核實，本版僅研究；**不得當成可訓練或發布 PASS**。